# 🧩 Building RAG Pipelines

This notebook builds a complete, end-to-end **Retrieval-Augmented Generation (RAG)** pipeline from scratch using LangChain: chunking a knowledge base, embedding it into a vector store, and wiring retrieval into an LLM chain in several progressively richer forms (plain answers, cited sources, graceful fallbacks, and structured output).

## Learning Objectives
In this notebook, you will learn:
1. **Vector store creation** - splitting raw text into chunks and embedding them into a `Chroma` vector store for semantic search.
2. **Basic RAG chains** - composing a retriever, prompt template, LLM, and output parser with LCEL (`|`) into a single runnable chain.
3. **Source-aware retrieval** - formatting retrieved documents so the LLM can cite which source(s) it used.
4. **Graceful fallback handling** - prompting the LLM to admit when the knowledge base doesn't contain an answer, instead of hallucinating.
5. **Structured RAG output** - using `with_structured_output` to return a validated Pydantic object (answer, confidence, sources, follow-up) instead of raw text.

## Prerequisites
- An `OPENAI_API_KEY` set in a `.env` file at the project root (loaded via `python-dotenv`).
- Familiarity with LangChain Expression Language (LCEL) `|` chain composition.
- Familiarity with basic RAG concepts (chunking, embeddings, retrieval) — see `04_Retrieval_and_RAG/Introduction_to_RAG/`.
- Packages: `langchain`, `langchain-openai`, `langchain-chroma`, `langchain-text-splitters`, `pydantic`.

---
## 🔧 Part 0: Setup

Before building any retrieval logic, we load environment variables, import the LangChain building blocks we'll reuse across every demo (prompts, runnables, output parsers, the `Chroma` vector store), and initialize the embedding model and LLM that every chain below shares.

### 📦 Environment & Imports

Imports are grouped standard library → third-party → LangChain. API keys come from a `.env` file via `python-dotenv` (`load_dotenv()`), not interactive prompts or hardcoded values.

In [9]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and API Key Loading
# ============================================================================
import tempfile
from typing import List

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

print("✅ Environment loaded and imports ready!")

✅ Environment loaded and imports ready!


### 📚 Sample Knowledge Base

We use a small markdown document about LangChain/LangGraph as the "knowledge base" for every demo below. In a real system this would be your document corpus (PDFs, wikis, tickets, etc.) — here it's kept small and self-contained so the retrieval behavior is easy to reason about.

In [10]:
# ============================================================================
# KNOWLEDGE BASE: Sample Document Content
# ============================================================================
KNOWLEDGE_BASE = """# LangChain Framework

LangChain is a framework for developing applications powered by language models.
It was created by Harrison Chase in October 2022.

## Core Components

1. **Models**: LangChain supports various LLM providers including OpenAI, Anthropic, and local models.
2. **Prompts**: Templates for structuring inputs to language models.
3. **Chains**: Sequences of calls to models and other components.
4. **Agents**: Systems that use LLMs to determine which actions to take.
5. **Memory**: Components for persisting state between chain/agent calls.

## LangGraph

LangGraph is a library for building stateful, multi-actor applications. Key features:
- State management
- Cycles and loops
- Human-in-the-loop
- Persistence

## Pricing

LangChain itself is open source and free. LangSmith (the observability platform) has a free tier and paid plans starting at $39/month.

## Getting Started

Install with: pip install langchain langchain-openai
Create your first chain in under 10 lines of code.
"""

# Embedding model used to vectorize chunks for similarity search
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

print(f"📄 Knowledge base loaded: {len(KNOWLEDGE_BASE)} characters")
print(f"🤖 Embedding model: {embeddings_model.model}")

📄 Knowledge base loaded: 1006 characters
🤖 Embedding model: text-embedding-3-small


### 🤖 LLM Initialization

We use `init_chat_model` to load the chat model that every RAG chain in this notebook shares. A low temperature (`0.2`) keeps answers grounded and consistent, which matters for RAG where we want the model to stick close to retrieved context rather than improvise.

In [11]:
# ============================================================================
# LLM INITIALIZATION: Chat Model for RAG Chains
# ============================================================================
llm = init_chat_model(model="gpt-4o-mini", temperature=0.2)

print(f"🤖 LLM initialized: {llm.model_name if hasattr(llm, 'model_name') else 'gpt-4o-mini'}")

🤖 LLM initialized: gpt-4o-mini


---
## 🧱 Part 1: Building the Vector Store

`create_kb()` turns the raw `KNOWLEDGE_BASE` markdown string into a searchable vector store: it splits the text into overlapping chunks, wraps it in a `Document`, and embeds the chunks into a fresh, temporary `Chroma` collection. Every demo below calls this function to get its own independent vector store.

In [12]:
# ============================================================================
# CREATE_KB: Build a Vector Store from the Knowledge Base
# ============================================================================
def create_kb():
    """Create a vector store from knowledge base."""
    # Split the knowledge base into chunks
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    doc = Document(
        page_content=KNOWLEDGE_BASE, metadata={"source": "langchain_knowledge_base.md"}
    )
    chunks = splitter.split_documents([doc])

    # Create a vector store from the chunks
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings_model,
        persist_directory=tempfile.mkdtemp(),
    )
    return vector_store

---
## 🔎 Part 2: Basic RAG Chain

`demo_basic_rag()` wires together the classic RAG pattern with LCEL: a `retriever` fetches the top-`k` similar chunks, `format_docs` joins them into a single context string, and the resulting `{context, question}` dict feeds a prompt → LLM → `StrOutputParser` chain. The prompt explicitly instructs the model to answer only from context and admit when it doesn't know.

In [13]:
# ============================================================================
# DEMO_BASIC_RAG: Retrieve, Augment, Generate
# ============================================================================
def demo_basic_rag():
    vector_store = create_kb()
    retriever = vector_store.as_retriever(
        search_type="similarity", search_kwargs={"k": 2}
    )

    # RAG prompt template
    prompt = ChatPromptTemplate.from_template(
        """Answer the question based only on the following context:

{context}

Question: {question}

Answer:
Make sure to answer in a concise manner, and if you don't know the answer, just say "I don't know."
"""
    )

    # Format retrieved docs into a single context string
    def format_docs(docs):
        return "\n\n".join([doc.page_content for doc in docs])

    # RAG chain: retrieve -> format -> prompt -> LLM -> parse
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    # Test the RAG chain
    questions = [
        "What is LangChain?",
        "Who created LangChain?",
        "What is LangGraph used for?",
    ]
    print("🔍 Basic RAG Demo:\n")
    for q in questions:
        answer = rag_chain.invoke(q)
        print(f"Q: {q}")
        print(f"A: {answer}\n")

---
## 📎 Part 3: RAG with Source Citations

`demo_rag_with_sources()` extends the basic pattern so the LLM can tell the user *which* retrieved chunks it relied on. `format_docs_with_sources` numbers each chunk and prefixes it with its `source` metadata, and the prompt asks the model to include those sources in its answer — a simple way to add traceability without a separate citation pipeline.

> **Key Insight**: Attaching metadata (like `source`) to documents at ingestion time is what makes citation possible later — the retriever passes metadata through untouched, so formatting decisions happen at query time.

In [14]:
# ============================================================================
# DEMO_RAG_WITH_SOURCES: Answer with Cited Sources
# ============================================================================
def demo_rag_with_sources():
    vectorstore = create_kb()
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    prompt = ChatPromptTemplate.from_template(
        """Answer the question based on the context below. Include which sources you used.

Context:
{context}

Question: {question}

Answer (include sources):"""
    )

    # Number each chunk and prefix it with its source metadata
    def format_docs_with_sources(docs):
        formatted = []
        for i, doc in enumerate(docs):
            source = doc.metadata.get("source", "unknown")
            formatted.append(f"[{i+1}] {source}:\n{doc.page_content}")
        return "\n\n".join(formatted)

    rag_chain = (
        {
            "context": retriever | format_docs_with_sources,
            "question": RunnablePassthrough(),
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    print("📎 RAG with Sources:\n")
    answer = rag_chain.invoke("What are the core components of LangChain?")
    print("Q: What are the core components?\n")
    print(f"A: {answer}")

---
## 🚫 Part 4: RAG with Fallback for Out-of-Scope Questions

A common RAG failure mode is the LLM confidently answering questions the knowledge base has no information about. `demo_rag_with_fallback()` guards against this by instructing the prompt to explicitly respond with a fixed "I don't have information about that" message whenever the answer isn't in the retrieved context, and tests it against both in-scope and out-of-scope questions.

In [15]:
# ============================================================================
# DEMO_RAG_WITH_FALLBACK: Admit When the Knowledge Base Doesn't Know
# ============================================================================
def demo_rag_with_fallback():
    vectorstore = create_kb()
    retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

    prompt = ChatPromptTemplate.from_template(
        """Answer the question based ONLY on the following context.
If the answer is not in the context, respond with: "I don't have information about that in my knowledge base."

Context:
{context}

Question: {question}

Answer:"""
    )

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    print("🚫 RAG with Fallback:\n")
    questions = [
        "What is the pricing for LangSmith?",  # In knowledge base
        "What is the stock price of OpenAI?",  # Not in knowledge base
        "How do I deploy LangChain to AWS?",  # Not in knowledge base
    ]
    for q in questions:
        answer = rag_chain.invoke(q)
        print(f"Q: {q}")
        print(f"A: {answer}\n")

---
## 🧾 Part 5: Structured RAG Output

Instead of returning free-form text, `demo_structured_rag()` defines a `RAGResponse` Pydantic model (answer, confidence, sources used, a suggested follow-up question) and binds it to the LLM with `with_structured_output`. The chain's final step then returns a validated Python object instead of a raw string — useful when a downstream system needs to consume the answer programmatically.

In [16]:
# ============================================================================
# DEMO_STRUCTURED_RAG: Validated, Structured RAG Response
# ============================================================================
def demo_structured_rag():
    """RAG with structured output."""
    vectorstore = create_kb()
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    class RAGResponse(BaseModel):
        """Structured RAG response."""

        answer: str = Field(description="The answer to the question")
        confidence: str = Field(description="high, medium, or low")
        sources_used: List[str] = Field(description="List of sources referenced")
        follow_up: str = Field(description="Suggested follow-up question")

    structured_llm = llm.with_structured_output(RAGResponse)

    prompt = ChatPromptTemplate.from_template(
        """Based on the context below, answer the question.

Context:
{context}

Question: {question}

Provide a structured response."""
    )

    def format_docs(docs):
        return "\n\n".join(
            f"[{doc.metadata.get('source', 'unknown')}]: {doc.page_content}"
            for doc in docs
        )

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | structured_llm
    )

    print("🧾 Structured RAG Demo:\n")
    result = rag_chain.invoke("What is LangGraph?")
    print(f"Answer: {result.answer}")
    print(f"Confidence: {result.confidence}")
    print(f"Sources: {result.sources_used}")
    print(f"Follow-up: {result.follow_up}")

---
## 🏋️ Part 6: Exercise — Document Q&A System

`exercise_document_qa()` packages the RAG pattern into a reusable `DocumentQA` class: given any text document, it splits, embeds, and builds a retrieval chain once in `__init__`, then exposes a simple `.ask(question)` method. The prompt also asks the model to self-rate its confidence (high/medium/low) for each answer. This demonstrates how the earlier standalone `demo_*` functions can be generalized into a small, reusable component.

In [21]:
# ============================================================================
# EXERCISE_DOCUMENT_QA: Reusable Document Q&A Class
# ============================================================================
def exercise_document_qa():
    """
    EXERCISE: Build a complete document Q&A system that:
    1. Takes a text document as input
    2. Splits and embeds it
    3. Allows multiple questions
    4. Returns answers with confidence scores
    """

    class DocumentQA:
        def __init__(self, document: str, source_name: str = "document"):
            # Split document
            splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
            doc = Document(page_content=document, metadata={"source": source_name})
            chunks = splitter.split_documents([doc])

            # Create vector store
            self.vectorstore = Chroma.from_documents(
                documents=chunks,
                embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
            )
            self.retriever = self.vectorstore.as_retriever(search_kwargs={"k": 3})

            # Create chain
            self.llm = init_chat_model(model="gpt-4o-mini", temperature=0.2)
            self.prompt = ChatPromptTemplate.from_template(
                """Answer based on the context. Rate your confidence (high/medium/low).

Context: {context}

Question: {question}

Format: [Confidence: X] Answer"""
            )

            def format_docs(docs):
                return "\n".join(d.page_content for d in docs)

            self.chain = (
                {
                    "context": self.retriever | format_docs,
                    "question": RunnablePassthrough(),
                }
                | self.prompt
                | self.llm
                | StrOutputParser()
            )

        def ask(self, question: str) -> str:
            return self.chain.invoke(question)

    # Test
    test_doc = """
    The Python programming language was created by Guido van Rossum.
    First released in 1991, Python emphasizes code readability.
    Python 3.12 was released in October 2023 with improved error messages.
    The language is named after Monty Python, not the snake.
    """

    qa = DocumentQA(test_doc, "python_facts")

    print("🏋️ Document Q&A System:\n")
    questions = [
        "Who created Python?",
        "When was Python 3.12 released?",
        "Why is Python named Python?",
    ]
    for q in questions:
        answer = qa.ask(q)
        print(f"Q: {q}")
        print(f"A: {answer}\n")

---
## ▶️ Running the Demos

The original script's `__main__` guard is kept verbatim below. Jupyter sets `__name__` to `"__main__"`, so this cell runs as-is; uncomment any of the commented-out lines to run that demo instead of (or in addition to) the exercise.

In [18]:
# ============================================================================
# RUN: Execute the Demos
# ============================================================================
if __name__ == "__main__":
    # demo_basic_rag()
    # demo_rag_with_sources()
    # demo_rag_with_fallback()
    # demo_structured_rag()
    exercise_document_qa()

🏋️ Document Q&A System:

Q: Who created Python?
A: [Confidence: high] Guido van Rossum

Q: When was Python 3.12 released?
A: [Confidence: High] Python 3.12 was released in October 2023.

Q: Why is Python named Python?
A: [Confidence: High] Python is named after Monty Python, not the snake.



---
## 📝 Summary

This notebook built a complete RAG pipeline in progressively richer stages, all sharing the same `create_kb()` vector store and `llm`.

### 1. Core Pipeline
- **`create_kb()`**: chunk a document and embed it into a `Chroma` vector store.
- **`demo_basic_rag()`**: the minimal retrieve → augment → generate chain with a "say I don't know" fallback instruction.

### 2. Richer Behaviors
- **`demo_rag_with_sources()`**: number and label retrieved chunks so the LLM can cite them.
- **`demo_rag_with_fallback()`**: enforce a fixed refusal message for out-of-scope questions instead of hallucinating.
- **`demo_structured_rag()`**: use `with_structured_output` + Pydantic to get a validated `answer`/`confidence`/`sources_used`/`follow_up` object.
- **`exercise_document_qa()`**: a reusable `DocumentQA` class that generalizes the pattern to any input document.

### Next Steps
- Explore query transformation techniques (multi-query, HyDE, RAG-Fusion) in `04_Retrieval_and_RAG/Query_Transformation_Techniques/`.
- Explore agentic and self-correcting RAG in `08_Advanced_RAG/RAG_with_LangGraph_Advanced/`.
- Try swapping `demo_rag_with_fallback()`'s hardcoded refusal for a retrieval-confidence check (e.g. score thresholding) instead of relying purely on prompt instructions.